# Notebook 12 — Baseline Comparison: Mean / KNN / MICE vs MIGA

## Motivation

MIGA minimises a **distributional distance** (Fr) between available rows X_A
and completed rows X_C.  Classical imputation methods minimise (explicitly or
implicitly) **pointwise squared error** (RMSE).  These are structurally
different objectives — so comparing RMSE across methods is an *apples-vs-oranges*
comparison, yet it is the standard in the missing-data literature.

This notebook makes that comparison explicit and frames MIGA's RMSE gaps
honestly: MIGA is not trying to minimise RMSE.

## Methods compared

| Method | Library | Key property |
|--------|---------|--------------|
| Mean   | `sklearn.SimpleImputer(strategy='mean')` | Baseline; minimises MSE to column mean |
| KNN    | `sklearn.KNNImputer(n_neighbors=5)` | Weighted average of k nearest complete rows |
| MICE   | `sklearn.IterativeImputer(max_iter=10)` | Iterative regression; closest to minimum-RMSE |
| MIGA   | Our reimplementation | Distributional distance minimisation |

**Expected finding (from theory):**
Van Buuren (2018, §2.6): "The minimum RMSE is achieved by regression imputation,
which suppresses variance."  MICE ≈ regression imputation → expected to win on RMSE.
MIGA may win on distributional fidelity (Fr) even when losing on RMSE.

**Note on baselines:** run `scripts/run_baselines.py` once (< 5 min) to generate
`results/12_baselines.json`.  MIGA results are loaded from the existing
`results/0{2-8}_*_results.json` files.

In [ ]:
import sys, os, json, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from miga import MIGA
from miga.fitness import FitnessEvaluator
from miga.statistics import compute_stats, pooled_std, relative_cov, minkowski_distance
from miga.data_utils import apply_mar, apply_mnar, auto_generators, compute_metrics, EXCLUDE_COLS
from miga.paper_results import (
    TABLE3_PARAMS, BENCHMARK_Q,
    TABLE4_RMSE, TABLE5_MAD, TABLE6_COD,
    METHODS, PERCENTAGES,
)

RESULTS_DIR = os.path.join("..", "results")
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Setup complete.")

## 1. Load Baseline Results

In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer

BASELINE_PATH = os.path.join(RESULTS_DIR, "12_baselines.json")
if not os.path.exists(BASELINE_PATH):
    print("Run scripts/run_baselines.py first to generate 12_baselines.json")
    baselines = {}
else:
    with open(BASELINE_PATH) as f:
        baselines = json.load(f)
    print(f"Loaded baselines for: {list(baselines.keys())}")

## 2. Load MIGA Results

In [ ]:
NB_META = {
    "Iris": "02", "Wine": "03", "Glass": "04",
    "Haberman": "05", "Wholesale": "06",
    "Cardio": "07", "Adult": "08",
}

miga_results = {}
for ds, num in NB_META.items():
    path = os.path.join(RESULTS_DIR, f"{num}_{ds}_results.json")
    if os.path.exists(path):
        with open(path) as f:
            miga_results[ds] = {int(k): v for k, v in json.load(f).items()}
        print(f"  Loaded MIGA: {ds}")
    else:
        print(f"  Missing MIGA: {path}  (run notebook {num} first)")

## 3. RMSE Comparison Table — All Datasets × All Methods

In [ ]:
all_ds = [d for d in NB_META if d in baselines and d in miga_results]

if not all_ds:
    print("No results available. Run scripts/run_baselines.py and notebooks 02–08 first.")
else:
    for pct in PERCENTAGES:
        print(f"\n{'='*70}")
        print(f"RMSE at {pct}% missing")
        print(f"{'='*70}")
        print(f"  {'Dataset':<12}  {'Mean':>8}  {'KNN':>8}  {'MICE':>8}  {'MIGA':>8}  {'MIGA best?':>12}")
        print("  " + "-" * 60)
        for ds in all_ds:
            b  = baselines[ds].get(str(pct), baselines[ds].get(pct, {}))
            mi = miga_results[ds].get(pct, {})
            vals = {
                "Mean": b.get("Mean", {}).get("rmse") or float("nan"),
                "KNN":  b.get("KNN",  {}).get("rmse") or float("nan"),
                "MICE": b.get("MICE", {}).get("rmse") or float("nan"),
                "MIGA": mi.get("rmse", float("nan")),
            }
            best_method = min(vals, key=lambda k: vals[k] if vals[k] == vals[k] else float("inf"))
            miga_rank   = sorted(vals, key=lambda k: vals[k] if vals[k] == vals[k] else float("inf")).index("MIGA") + 1
            rank_str    = f"rank {miga_rank}/4"
            print(f"  {ds:<12}  {vals['Mean']:>8.4f}  {vals['KNN']:>8.4f}  {vals['MICE']:>8.4f}  {vals['MIGA']:>8.4f}  {rank_str:>12}  [best: {best_method}]")

## 4. RMSE Ratio: MIGA / MICE

In [ ]:
if all_ds:
    print("MIGA/MICE RMSE ratio  (> 1 = MICE wins, < 1 = MIGA wins)")
    print()
    print(f"  {'Dataset':<12}  {'30%':>8}  {'40%':>8}  {'50%':>8}  {'60%':>8}  {'Avg':>8}")
    print("  " + "-" * 55)
    for ds in all_ds:
        row = []
        for pct in PERCENTAGES:
            b  = baselines[ds].get(str(pct), baselines[ds].get(pct, {}))
            mi = miga_results[ds].get(pct, {})
            mice_rmse = (b.get("MICE", {}).get("rmse") or float("nan"))
            miga_rmse = mi.get("rmse", float("nan"))
            if mice_rmse and mice_rmse == mice_rmse and miga_rmse == miga_rmse:
                row.append(miga_rmse / mice_rmse)
            else:
                row.append(float("nan"))
        avg = sum(r for r in row if r == r) / max(1, sum(1 for r in row if r == r))
        cells = ["★" if r < 1.0 else f"{r:.2f}x" for r in row]
        print(f"  {ds:<12}  {'  '.join(f'{c:>8}' for c in cells)}  {avg:>8.2f}x")
    print()
    print("  ★ = MIGA beats MICE on RMSE (rare — MICE optimizes RMSE directly)")

## 5. The Honest Framing

MICE wins on RMSE because it **explicitly minimises squared error** via iterative
regression.  MIGA is not designed to minimise RMSE — it minimises the distributional
distance Fr between X_A and X_C.

**Van Buuren (2018, §2.6):**
> "The minimum mean squared error is achieved by regression imputation, which
> suppresses variance and produces biased statistical inference."

The implication: MICE imputes "average" values (near the regression line), which
scores well on RMSE but *underestimates the true variance* of the missing values.
MIGA imputes from the empirical distribution of each feature, preserving variance
at the cost of pointwise accuracy.

**When MIGA is preferable to MICE:**
- When downstream analyses require the correct *marginal distribution*
  (histograms, quantile estimation, distributional tests)
- When data is non-Gaussian and regression imputation is misspecified
- When the goal is to *recover the joint distribution* rather than point estimates

**When MICE is preferable:**
- When the goal is to minimise prediction error on specific values
- When downstream analyses require minimum-variance estimates

## 6. Visualisation — RMSE by Method

In [ ]:
if all_ds:
    pct = 30
    methods = ["Mean", "KNN", "MICE", "MIGA"]
    cols_m = {"Mean": "tab:gray", "KNN": "tab:green", "MICE": "tab:orange", "MIGA": "tab:blue"}
    x = np.arange(len(all_ds))
    width = 0.2

    fig, ax = plt.subplots(figsize=(12, 5))
    for i, method in enumerate(methods):
        rmse_vals = []
        for ds in all_ds:
            b  = baselines[ds].get(str(pct), baselines[ds].get(pct, {}))
            mi = miga_results[ds].get(pct, {})
            if method == "MIGA":
                rmse_vals.append(mi.get("rmse", float("nan")))
            else:
                rmse_vals.append((b.get(method, {}).get("rmse") or float("nan")))
        ax.bar(x + (i - 1.5) * width, rmse_vals, width,
               label=method, color=cols_m[method], alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(all_ds, rotation=20, ha="right")
    ax.set_ylabel("NRMSE")
    ax.set_title(f"Imputation RMSE at {pct}% missing — Mean / KNN / MICE / MIGA")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "12_rmse_comparison.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: results/12_rmse_comparison.png")

## 7. CoD Comparison — Does MIGA Win on Any Metric?

In [ ]:
if all_ds:
    pct = 30
    print(f"CoD at {pct}% missing  (higher = better):")
    print(f"  {'Dataset':<12}  {'Mean':>8}  {'KNN':>8}  {'MICE':>8}  {'MIGA':>8}  {'MIGA best?'}")
    print("  " + "-" * 60)
    for ds in all_ds:
        b  = baselines[ds].get(str(pct), baselines[ds].get(pct, {}))
        mi = miga_results[ds].get(pct, {})
        vals = {
            "Mean": b.get("Mean", {}).get("cod") or float("nan"),
            "KNN":  b.get("KNN",  {}).get("cod") or float("nan"),
            "MICE": b.get("MICE", {}).get("cod") or float("nan"),
            "MIGA": mi.get("cod", float("nan")),
        }
        best = max(vals, key=lambda k: vals[k] if vals[k] == vals[k] else float("-inf"))
        miga_best = "★" if best == "MIGA" else ""
        print(f"  {ds:<12}  {vals['Mean']:>8.4f}  {vals['KNN']:>8.4f}  {vals['MICE']:>8.4f}  {vals['MIGA']:>8.4f}  {miga_best}")

## 8. Summary

In [ ]:
if all_ds:
    print("=" * 60)
    print("SUMMARY — Method Comparison")
    print("=" * 60)
    print()
    print("RMSE ranking (across all datasets and percentages):")
    method_avg = {m: [] for m in ["Mean", "KNN", "MICE", "MIGA"]}
    for ds in all_ds:
        for pct in PERCENTAGES:
            b  = baselines[ds].get(str(pct), baselines[ds].get(pct, {}))
            mi = miga_results[ds].get(pct, {})
            method_avg["Mean"].append(b.get("Mean", {}).get("rmse") or float("nan"))
            method_avg["KNN"].append(b.get("KNN",  {}).get("rmse") or float("nan"))
            method_avg["MICE"].append(b.get("MICE", {}).get("rmse") or float("nan"))
            method_avg["MIGA"].append(mi.get("rmse", float("nan")))
    for method, vals in sorted(method_avg.items(),
                               key=lambda kv: np.nanmean(kv[1])):
        avg = np.nanmean(vals)
        print(f"  {method:<6}: avg RMSE = {avg:.4f}")
    print()
    print("Interpretation:")
    print("  MICE wins on RMSE because it explicitly minimises squared error.")
    print("  MIGA optimises distributional distance (Fr), not RMSE.")
    print("  MIGA is preferable when variance preservation matters more than pointwise accuracy.")